In [1]:
import cadquery as cq
import math

In [2]:
# ----------------------------
# Parameters
# ----------------------------
output_file = "snake_scales_jnb.stl"

num_scales = 5

# Base plate
plate_length = 300.0  # along Y
plate_width  = 120.0  # along X
plate_thickness = 2.0

# Scale parameters
base_width = 10.0
base_length = 50.0
tip_width = 6.0
tip_length = 10.0
path_length = plate_width - plate_width/4
attack_angle = 30.0
num_sections = 8
curvature_factor = 1.3

# Margins
margin_outer = 10.0 
margin_inner = 10.0 
x_offset_scale = 60.0

In [3]:
def create_scale(base_width, base_length, tip_width, tip_length, path_length, attack_angle, num_sections, curvature_factor):
    angle_rad = math.radians(attack_angle)
    
    y_end = path_length * math.cos(angle_rad)
    z_end = path_length * math.sin(angle_rad)
    
    y_start = base_length / 2
    
    spine = (
        cq.Workplane("YZ")
        .moveTo(y_start, 0)
        .threePointArc(
            (y_start + y_end * 0.5, z_end * curvature_factor),
            (y_start + y_end, z_end)
        )
        .val()
    )
    
    def lerp(a, b, t):
        return a + (b - a) * t
    
    profiles = []
    
    Z = cq.Vector(0, 0, 1)
    Y = cq.Vector(0, 1, 0)
    X = cq.Vector(1, 0, 0)
    
    # Generate trapezoid profiles
    for i in range(num_sections):
        t = i / (num_sections - 1)
    
        point = spine.positionAt(t)
    
        t_rot = t**1.5
        normal = (Z.multiply(1 - t_rot) + Y.multiply(t_rot)).normalized()
    
        x_dir = X
        y_dir = normal.cross(x_dir).normalized()
    
        plane = cq.Plane(origin=point, xDir=x_dir, normal=normal)
    
        width = lerp(base_width, tip_width, t)
    
        bottom_len = base_length * (1 - 0.4 * t)
        top_len = lerp(base_length * 0.6, tip_length, t)
    
        pts = [
            (-bottom_len/2, -width/2),
            ( bottom_len/2, -width/2),
            ( top_len/2,    width/2),
            (-top_len/2,    width/2)
        ]
    
        wire = cq.Workplane(plane).polyline(pts).close().val()
        profiles.append(wire)
    
    scale = cq.Solid.makeLoft(profiles, ruled=False)
    return scale

In [4]:
# ----------------------------
# Create base plate
# ----------------------------
plate = cq.Workplane("XY").rect(plate_width, plate_length).extrude(plate_thickness)

# ----------------------------
# Compute spacing along Y
# ----------------------------
available_length = plate_length - margin_outer - margin_inner - num_scales * base_length
if num_scales > 1:
    spacing = available_length / (num_scales - 1)
else:
    spacing = 0.0

if spacing < 0:
    raise ValueError("Scales do not fit on plate. Increase plate_length or reduce scale size/number.")

# ----------------------------
# Place scales along Y
# ----------------------------
scales = cq.Workplane("XY")
y = -plate_length / 2 + margin_outer + base_length / 2  # start first scale

for i in range(num_scales):
    scale = create_scale(
        base_width,
        base_length,
        tip_width,
        tip_length,
        path_length,
        attack_angle,
        num_sections,
        curvature_factor
    )

    # Rotate so spine points along X
    scale = scale.rotate((0,0,0), (0,0,1), 90)

    # Translate to correct Y position (preserve X curvature)
    scale = scale.translate((x_offset_scale, y, plate_thickness))
    scales = scales.add(scale)

    # Increment Y for next scale
    y += base_length + spacing

# ----------------------------
# Combine scales and plate
# ----------------------------
result = plate.union(scales)
cq.exporters.export(result, output_file)
print("STL exported:", output_file)
display(result)

STL exported: snake_scales_jnb.stl
